In [49]:
import sys
import os
import time
import pandas as pd
import glob

PROJECT_ROOT = r"C:\Users\Pablo Miller\proyectos\pabs-nba-analytics-dashboard"
sys.path.insert(0, PROJECT_ROOT)

from nba_api.stats.endpoints import shotchartdetail

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\Pablo Miller\proyectos\pabs-nba-analytics-dashboard


In [50]:
import glob
import os

raw_dir = os.path.join(PROJECT_ROOT, "data", "raw")
pattern = os.path.join(raw_dir, "season_stats_*.csv")

all_files = sorted(glob.glob(pattern))

season_files = []

for file in all_files:
    season_str = os.path.basename(file).replace("season_stats_", "").replace(".csv", "")
    year_start = int(season_str.split("-")[0])

    # ONLY keep seasons from 2010 onward
    if year_start >= 2010:
        season_files.append(file)

print("Found modern seasons:", len(season_files))
season_files

Found modern seasons: 16


['C:\\Users\\Pablo Miller\\proyectos\\pabs-nba-analytics-dashboard\\data\\raw\\season_stats_2010-11.csv',
 'C:\\Users\\Pablo Miller\\proyectos\\pabs-nba-analytics-dashboard\\data\\raw\\season_stats_2011-12.csv',
 'C:\\Users\\Pablo Miller\\proyectos\\pabs-nba-analytics-dashboard\\data\\raw\\season_stats_2012-13.csv',
 'C:\\Users\\Pablo Miller\\proyectos\\pabs-nba-analytics-dashboard\\data\\raw\\season_stats_2013-14.csv',
 'C:\\Users\\Pablo Miller\\proyectos\\pabs-nba-analytics-dashboard\\data\\raw\\season_stats_2014-15.csv',
 'C:\\Users\\Pablo Miller\\proyectos\\pabs-nba-analytics-dashboard\\data\\raw\\season_stats_2015-16.csv',
 'C:\\Users\\Pablo Miller\\proyectos\\pabs-nba-analytics-dashboard\\data\\raw\\season_stats_2016-17.csv',
 'C:\\Users\\Pablo Miller\\proyectos\\pabs-nba-analytics-dashboard\\data\\raw\\season_stats_2017-18.csv',
 'C:\\Users\\Pablo Miller\\proyectos\\pabs-nba-analytics-dashboard\\data\\raw\\season_stats_2018-19.csv',
 'C:\\Users\\Pablo Miller\\proyectos\\pabs-nba

In [51]:
def download_shots(player_id, season, season_type):
    sc = shotchartdetail.ShotChartDetail(
        team_id=0,
        player_id=player_id,
        season_nullable=season,
        season_type_all_star=season_type,
        context_measure_simple="FGA"
    )
    return sc.get_data_frames()[0]

In [52]:
shots_dir = os.path.join(PROJECT_ROOT, "data", "shots")
os.makedirs(shots_dir, exist_ok=True)

In [53]:
for file in season_files:
    # Extract season string from filename
    season = os.path.basename(file).replace("season_stats_", "").replace(".csv", "")
    print(f"\n=== Processing season {season} ===")

    # Load season stats CSV
    df = pd.read_csv(file)

    # Ensure PLAYER_ID exists
    if "PLAYER_ID" not in df.columns:
        raise ValueError(f"PLAYER_ID column missing in {file}")

    # Get unique players
    players = df[["PLAYER_ID", "PLAYER_NAME"]].drop_duplicates()

    # Create season folders
    season_dir = os.path.join(shots_dir, season)
    regular_dir = os.path.join(season_dir, "regular")
    playoffs_dir = os.path.join(season_dir, "playoffs")
    os.makedirs(regular_dir, exist_ok=True)
    os.makedirs(playoffs_dir, exist_ok=True)

    # Loop through players
    for _, row in players.iterrows():
        pid = row["PLAYER_ID"]
        name = row["PLAYER_NAME"]

        for stype, folder_dir in [
            ("Regular Season", regular_dir),
            ("Playoffs", playoffs_dir),
        ]:
            path = os.path.join(folder_dir, f"{pid}.csv")

            # Skip if already downloaded
            if os.path.exists(path):
                continue

            try:
                df_shots = download_shots(pid, season, stype)
                df_shots.to_csv(path, index=False)

                print(f"Success: Downloaded {name}'s {season} {stype} data")

                time.sleep(1.2)  # avoid rate limits

            except Exception as e:
                print(f"Failed: {name} {season} {stype} → {e}")
                time.sleep(3)


=== Processing season 2010-11 ===
Success: Downloaded AJ Price's 2010-11 Regular Season data
Success: Downloaded AJ Price's 2010-11 Playoffs data
Success: Downloaded Aaron Brooks's 2010-11 Regular Season data
Success: Downloaded Aaron Brooks's 2010-11 Playoffs data
Success: Downloaded Aaron Gray's 2010-11 Regular Season data
Success: Downloaded Aaron Gray's 2010-11 Playoffs data
Success: Downloaded Acie Law's 2010-11 Regular Season data
Success: Downloaded Acie Law's 2010-11 Playoffs data
Success: Downloaded Al Harrington's 2010-11 Regular Season data
Success: Downloaded Al Harrington's 2010-11 Playoffs data
Success: Downloaded Al Horford's 2010-11 Regular Season data
Success: Downloaded Al Horford's 2010-11 Playoffs data
Success: Downloaded Al Jefferson's 2010-11 Regular Season data
Success: Downloaded Al Jefferson's 2010-11 Playoffs data
Success: Downloaded Al Thornton's 2010-11 Regular Season data
Success: Downloaded Al Thornton's 2010-11 Playoffs data
Success: Downloaded Al-Farouq

KeyboardInterrupt: 